In [2]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import gc
from scipy import stats
from statsmodels.stats.multitest import multipletests
from cmldask import CMLDask as da
from dask.distributed import wait, as_completed

from ptsa.data.filters import ButterworthFilter, MorletWaveletFilter
import cmlreaders as cml

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('display.max_columns', None)

In [3]:
EXP = 'CourierReinstate1'
SUBJECTS = ['LTP564', 'LTP565', 'LTP566', 'LTP567', 'LTP568', 'LTP569', 'LTP571', 'LTP572', 'LTP573',
            'LTP574', 'LTP575', 'LTP576', 'LTP577', 'LTP578', 'LTP579', 'LTP580', 'LTP581', 'LTP583',
            'LTP584', 'LTP585', 'LTP586', 'LTP587', 'LTP588', 'LTP589', 'LTP590', 'LTP591', 'LTP592', 
            'LTP593', 'LTP594', 'LTP595', 'LTP596', 'LTP597', 'LTP598', 'LTP599', 'LTP600', 'LTP601', 
            'LTP602', 'LTP603', 'LTP604', 'LTP605']

REL_START, REL_STOP = 300, 1300
BUFFER_MS = 1000
WIDTH = 6

FREQS = np.logspace(np.log10(2), np.log10(100), 46)
NOTCH_BAND = (58., 62.)
BATCH_EVENTS = 64
ROI_ORDER = ['LAI','RAI','LAS','RAS','LPS','RPS','LPI','RPI']

In [4]:
def assign_roi(channel):
    roi_dict = {
        'LAS':['C24','C25','D2','D3','D4','D11','D12','D13'],
        'LAI':['C31','C32','D5','D6','D9','D10','D21','D22'],
        'LPS':['D29','A5','A6','A7','A8','A17','A18'],
        'LPI':['D30','D31','A9','A10','A11','A15','A16'],
        'RAS':['B30','B31','B32','C2','C3','C4','C11','C12'],
        'RAI':['B24','B25','B28','B29','C5','C6','C9','C10'],
        'RPS':['A30','A31','A32','B3','B4','B5','B13'],
        'RPI':['A28','A29','B6','B7','B8','B11','B12'],
    }
    for roi, chans in roi_dict.items():
        if channel in chans:
            return roi
    return None

In [5]:
df_idx = cml.get_data_index('ltp').query("experiment == @EXP").copy()
if SUBJECTS is not None:
    df_idx = df_idx[df_idx['subject'].isin(SUBJECTS)]
subjects = sorted(df_idx.subject.unique())
print(f'Found {len(subjects)} subjects for experiment "{EXP}".')

Found 40 subjects for experiment "CourierReinstate1".


In [7]:
def process_subject_stats(subject, EXP=EXP, batch=BATCH_EVENTS):
    """
    Returns DataFrame: subject, session, trial, serialpos, freq, recalled, power
    One row = one WORD event at one freq
    Power averaged across channels
    """
    sessions = cml.get_data_index('ltp').query("experiment == @EXP and subject == @subject").session.unique()

    dfs = []
    freq_vec = np.array(FREQS)

    for sess in sessions:
        try:
            reader = cml.CMLReader(subject=subject, experiment=EXP, session=sess)
            evs_all = reader.load('task_events')
            evs_all = evs_all[(evs_all['type']=='WORD') & (evs_all['eegoffset']>=0)]
            if evs_all.empty:
                continue

            for start in range(0, len(evs_all), batch):
                evs = evs_all.iloc[start:start+batch]
                eeg = reader.load_eeg(evs, rel_start=-BUFFER_MS, rel_stop=REL_STOP+BUFFER_MS, clean='LCF').to_ptsa()

                # Notch filter
                eeg = ButterworthFilter(freq_range=list(NOTCH_BAND), filt_type='stop', order=4).filter(eeg)

                # Wavelet power
                mw = MorletWaveletFilter(freqs=freq_vec, width=WIDTH, output='power', complete=True)
                pwr = mw.filter(eeg).sel(time=slice(REL_START, REL_STOP)).mean(dim='time')
                p = pwr.transpose('event', 'channel', 'frequency').values
                p_event_freq = p.mean(axis=1)  # averaged across channels
                
                # Event-level metadata
                meta = evs[['subject', 'session', 'trial', 'serialpos', 'recalled']]
                meta['recalled'] = meta['recalled'].astype(int)
                
                power_df = pd.DataFrame(p_event_freq, columns=freq_vec, index=meta.index)
                batch_df = pd.concat([meta.reset_index(drop=True), power_df.reset_index(drop=True)], axis=1)
                batch_long = batch_df.melt(id_vars=['subject', 'session', 'trial', 'serialpos', 'recalled'], var_name='freq', value_name='power')
                dfs.append(batch_long)

                del eeg, pwr, p, p_event_freq, power_df, batch_df, batch_long
                gc.collect()

        except Exception as e:
            print(f"[WARN] {subject} session {sess} skipped: {e}")
            continue

    if not dfs:
        return pd.DataFrame(columns=['subject', 'session', 'trial', 'serialpos', 'freq', 'recalled', 'power'])
    
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(['subject', 'session', 'trial', 'serialpos', 'freq']).reset_index(drop=True)
    
    return df

In [6]:
client = da.new_dask_client_slurm(
    job_name="encoding",
    memory_per_job="50GB",
    max_n_jobs=10,
    queue="RAM",
    local_directory="",
    log_directory="dask_logs"
)

Unique port for joycerose14 is 51609
{'dashboard_address': ':51609'}
To view the dashboard, run: 
`ssh -fN joycerose14@rhino2.psych.upenn.edu -L 8000:192.168.86.104:51609` in your local computer's terminal (NOT rhino) 
and then navigate to localhost:8000 in your browser


In [7]:
futures = [client.submit(process_subject_stats, s, EXP, BATCH_EVENTS) for s in subjects]

In [8]:
all_dfs, n_done, n_fail = [], 0, 0
for fut in as_completed(futures):
    try:
        df_sub = fut.result()
        all_dfs.append(df_sub)
        n_done += 1
        sub = df_sub['subject'].iat[0] if not df_sub.empty else 'EMPTY_SUBJECT'
        print(f"[DONE] {n_done}/{len(subjects)} :: {sub} (rows={len(df_sub)})")
    except Exception as e:
        n_fail += 1
        print(f"[ERR] Future failed ({n_fail} fails): {e}")

[DONE] 1/40 :: LTP565 (rows=13800)
[DONE] 2/40 :: LTP564 (rows=6900)
[DONE] 3/40 :: LTP567 (rows=27692)
[DONE] 4/40 :: LTP569 (rows=26082)
[DONE] 5/40 :: LTP566 (rows=13110)
[DONE] 6/40 :: LTP574 (rows=37950)
[DONE] 7/40 :: LTP572 (rows=37950)
[DONE] 8/40 :: LTP583 (rows=31050)
[DONE] 9/40 :: LTP578 (rows=31050)
[DONE] 10/40 :: LTP580 (rows=31050)
[DONE] 11/40 :: EMPTY_SUBJECT (rows=0)
[DONE] 12/40 :: LTP576 (rows=37950)
[DONE] 13/40 :: LTP585 (rows=37950)
[DONE] 14/40 :: LTP586 (rows=31372)
[DONE] 15/40 :: LTP577 (rows=10350)
[DONE] 16/40 :: LTP568 (rows=37950)
[DONE] 17/40 :: LTP588 (rows=37950)
[DONE] 18/40 :: LTP573 (rows=37950)
[DONE] 19/40 :: LTP571 (rows=37950)
[DONE] 20/40 :: LTP581 (rows=31050)
[DONE] 21/40 :: LTP595 (rows=37950)
[DONE] 22/40 :: LTP592 (rows=10350)
[DONE] 23/40 :: LTP587 (rows=37950)
[DONE] 24/40 :: LTP594 (rows=29118)
[DONE] 25/40 :: LTP593 (rows=10350)
[DONE] 26/40 :: LTP590 (rows=18676)
[DONE] 27/40 :: LTP584 (rows=33994)
[DONE] 28/40 :: LTP575 (rows=37950)

In [10]:
client.shutdown()

In [11]:
df_all = pd.concat(all_dfs, ignore_index=True)
df_all

,subject,session,trial,serialpos,recalled,freq,power
0,LTP565,3,0,1,1,2.0,2.510806e-05
1,LTP565,3,0,1,1,2.181649,2.177011e-05
2,LTP565,3,0,1,1,2.379796,1.618050e-05
3,LTP565,3,0,1,1,2.59594,1.344866e-05
4,LTP565,3,0,1,1,2.831715,1.261803e-05
...,...,...,...,...,...,...,...
1151145,LTP603,5,9,15,1,70.628575,7.934283e-07
1151146,LTP603,5,9,15,1,77.043381,7.937953e-07
1151147,LTP603,5,9,15,1,84.040809,7.130217e-07
1151148,LTP603,5,9,15,1,91.673774,5.959702e-07


In [12]:
df_all.to_csv('csv/encoding_df.csv', index=False)